# Run All Instructor Setup Scripts

This notebook runs all setup scripts in order:  

0. Set up all tables
1. Knowledge Assistant setup
2. Genie space setup
3. Vega-Lite UC Function setup
3b. You.com Web Search UC Functions setup
3c. OntoBricks GraphRAG UC functions setup (optional)
4. Multi-Agent Supervisor setup (auto-discovers KA, Genie, and UC functions)
5. Deploy Chatbot App with Lakebase

In [0]:
import os

# Clear VS Code Databricks extension env vars (temp workaround)
# %run ./clear_db_env

# Use the pre-configured serverless spark session (already available in serverless notebooks)
print(f"Connected to Databricks serverless - Spark version: {spark.version}")

# Set DATABRICKS_HOST from notebook context for downstream WorkspaceClient() calls
from databricks.sdk.runtime import dbutils
notebook_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
# browserHostName() is only set in interactive UI runs; spark.conf works in jobs too.
try:
    host = notebook_context.browserHostName().get()
except Exception:
    host = None
if not host:
    host = spark.conf.get("spark.databricks.workspaceUrl")
os.environ["DATABRICKS_HOST"] = f"https://{host}"
print(f"Workspace: https://{host}")

In [0]:
# Step 0: Set up tables (creates schema, volume, uploads data, builds tables)
result = dbutils.notebook.run("./00_tables_setup", 600)
print(f"✅ volume and table setup completed: {result}")

In [0]:
# Step 1: Create Knowledge Assistant
result = dbutils.notebook.run("./01_instructor_setup_ka", 600)
print(f"✅ Knowledge Assistant setup completed: {result}")

In [0]:
# Step 2: Create Genie Space
result = dbutils.notebook.run("./02_instructor_setup_genie", 600)
print(f"✅ Genie Space setup completed: {result}")

In [0]:
# Step 3: Create Vega-Lite UC Function
result = dbutils.notebook.run("./03_create_vegalite_uc_function_simple", 600)
print(f"✅ Vega-Lite UC Function setup completed: {result}")

In [0]:
# Step 3b: Create You.com Web Search UC Functions
result = dbutils.notebook.run("./03b_create_youdotcom_uc_functions", 600)
print(f"You.com UC Functions setup completed: {result}")

In [0]:
# Step 3c: GraphRAG UC functions (Optional)
# Builds graphrag_* tables and get_company_summary / compare_companies for the Supervisor.
try:
    result = dbutils.notebook.run("./05b_create_ontobricks_graphrag_OPTIONAL", 600)
    print(f"\u2705 OntoBricks GraphRAG UC functions setup completed: {result}")
except Exception as e:
    print(f"\u26a0\ufe0f OntoBricks GraphRAG setup skipped or failed: {e}")


In [0]:
# Step 4: Create Multi-Agent Supervisor
result = dbutils.notebook.run("./04_instructor_setup_sa", 600)
print(f"✅ Multi-Agent Supervisor setup completed: {result}")

In [0]:
# Step 5: Deploy Chatbot App
result = dbutils.notebook.run("./06_deploy_chatbot_app", 600, {"host": host})
print(f"✅ Chatbot App deployment completed: {result}")